# Random Forest — evaluation figures

Reads the per-fold test predictions from `rf_results/predictions.npz` (written by
`random_forest_model.ipynb`) and produces two PNGs in `rf_results/`:

- `confusion_matrix.png` — pooled, row-normalised
- `per_class_metrics.png` — F1, accuracy and balanced accuracy per activity

Run the cells in order.

In [ ]:
# ============================================================================
#  Random Forest evaluation figures - load predictions and compute metrics
# ----------------------------------------------------------------------------
#  Reads the per-fold test predictions written by random_forest_model.ipynb and
#  pools them into a single evaluation. Every user is tested exactly once across
#  the five folds, so the pooled result is one estimate over all 56 users -
#  strictly better than averaging five fold-level scores, whose test sets range
#  from 133k to 371k segments.
# ============================================================================
import json
import os

import numpy as np
from sklearn.metrics import (balanced_accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score,
                             precision_recall_fscore_support)

RESULTS_DIR = "rf_results"
FIG_DIR = "rf_results"
CLASS_NAMES = ["Lying down", "Sitting", "Walking", "Running",
               "Bicycling", "Standing in place", "Standing and moving"]
N_CLASSES = len(CLASS_NAMES)

_d = np.load(os.path.join(RESULTS_DIR, "predictions.npz"), allow_pickle=True)
y_true = np.concatenate([_d[f"y_true_{i}"] for i in range(5)])
y_pred = np.concatenate([_d[f"y_pred_{i}"] for i in range(5)])

cm = confusion_matrix(y_true, y_pred, labels=range(N_CLASSES))
cm_row = cm / cm.sum(axis=1, keepdims=True)          # normalised by TRUE class

prec, rec, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=range(N_CLASSES), zero_division=0)

# per-class one-vs-rest accuracy and balanced accuracy
ovr_acc, ovr_bal, spec = np.zeros(N_CLASSES), np.zeros(N_CLASSES), np.zeros(N_CLASSES)
total = len(y_true)
for k in range(N_CLASSES):
    tp = cm[k, k]
    fn = support[k] - tp
    fp = cm[:, k].sum() - tp
    tn = total - tp - fn - fp
    ovr_acc[k] = (tp + tn) / total
    spec[k] = tn / (tn + fp)
    ovr_bal[k] = (rec[k] + spec[k]) / 2

POOLED = {"accuracy": float((y_true == y_pred).mean()),
          "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
          "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
          "kappa": float(cohen_kappa_score(y_true, y_pred))}

print(f"pooled over 5 folds: {total:,} test segments\n")
print("  " + "  ".join(f"{k} {v:.4f}" for k, v in POOLED.items()))
print(f"\n{'class':22s} {'prec':>6} {'recall':>7} {'F1':>6} "
      f"{'OvR acc':>8} {'OvR bal':>8} {'support':>10}")
for k in range(N_CLASSES):
    print(f"{CLASS_NAMES[k]:22s} {prec[k]:6.3f} {rec[k]:7.3f} {f1[k]:6.3f} "
          f"{ovr_acc[k]:8.3f} {ovr_bal[k]:8.3f} {support[k]:10,}")

json.dump({"pooled": POOLED,
           "per_class": {CLASS_NAMES[k]: {"precision": float(prec[k]),
                                          "recall": float(rec[k]),
                                          "f1": float(f1[k]),
                                          "ovr_accuracy": float(ovr_acc[k]),
                                          "ovr_balanced_accuracy": float(ovr_bal[k]),
                                          "support": int(support[k])}
                         for k in range(N_CLASSES)}},
          open(os.path.join(RESULTS_DIR, "pooled_metrics.json"), "w"), indent=1)

In [ ]:
# ============================================================================
#  Figure 1 - pooled confusion matrix
# ----------------------------------------------------------------------------
#  Normalised by TRUE class (rows sum to 100%), because raw counts would render
#  Running's 4,985 segments invisible beside Sitting's 588,134. The diagonal is
#  therefore per-class recall. Raw counts are kept as a second line in each cell.
# ============================================================================
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# sequential blue ramp, light -> dark (palette steps 100..700)
SEQ = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
       "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
CMAP = LinearSegmentedColormap.from_list("seq_blue", SEQ)
SURFACE, INK, INK_2, INK_3 = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8983"

CLASS_SHORT = ["Lying\ndown", "Sitting", "Walking", "Running",
               "Bicycling", "Standing\nin place", "Standing &\nmoving"]

mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9.5,
                     "figure.dpi": 110})

fig, ax = plt.subplots(figsize=(10.2, 8.6))
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)
fig.subplots_adjust(top=0.845, left=0.20, right=0.94, bottom=0.11)

im = ax.imshow(cm_row, cmap=CMAP, vmin=0, vmax=1, aspect="equal")

# 2px surface gap between cells
ax.set_xticks(np.arange(-.5, N_CLASSES, 1), minor=True)
ax.set_yticks(np.arange(-.5, N_CLASSES, 1), minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=2.4)
ax.tick_params(which="minor", length=0)

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        v = cm_row[i, j]
        txt = INK if v < 0.45 else "#ffffff"           # flip on dark cells
        ax.text(j, i - 0.11, f"{v*100:.1f}%", ha="center", va="center",
                color=txt, fontsize=11.5 if i == j else 10.5,
                fontweight="bold" if i == j else "normal")
        ax.text(j, i + 0.24, f"{cm[i, j]:,}", ha="center", va="center",
                color=txt, fontsize=7.5, alpha=.60)

ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(CLASS_SHORT, fontsize=9, color=INK_2, linespacing=1.35)
ax.set_yticklabels([f"{n}\n{s:,}" for n, s in zip(CLASS_NAMES, support)],
                   fontsize=9, color=INK_2, linespacing=1.45)
ax.tick_params(axis="both", length=0, pad=7)

ax.set_xlabel("Predicted", fontsize=10.5, color=INK_2, labelpad=14)
ax.set_ylabel("Actual  ·  segments in class", fontsize=10.5, color=INK_2, labelpad=14)
for s in ax.spines.values():
    s.set_visible(False)

fig.text(0.055, 0.955, "Random Forest — pooled confusion matrix",
         fontsize=16, color=INK, fontweight="bold", va="top")
fig.text(0.055, 0.905,
         f"5 folds concatenated · {len(y_true):,} test segments · each row normalised "
         f"by true class, so the diagonal is recall",
         fontsize=9.6, color=INK_3, va="top")

cb = fig.colorbar(im, ax=ax, fraction=0.030, pad=0.035)
cb.set_label("share of the true class", fontsize=8.8, color=INK_3, labelpad=10)
cb.ax.tick_params(labelsize=8, length=0, colors=INK_3)
cb.outline.set_visible(False)
cb.set_ticks([0, .25, .5, .75, 1])
cb.set_ticklabels(["0%", "25%", "50%", "75%", "100%"])

out = os.path.join(FIG_DIR, "confusion_matrix.png")
fig.savefig(out, dpi=200, facecolor=SURFACE)
print(f"saved {out}  ({os.path.getsize(out)/1000:.0f} KB)")
plt.close(fig)

In [ ]:
# ============================================================================
#  Figure 2 - per-class F1, accuracy and balanced accuracy
# ----------------------------------------------------------------------------
#  Accuracy and balanced accuracy are defined per class in the one-vs-rest sense
#  (this class against all others), since both are otherwise whole-model numbers.
#
#  Read the accuracy bars with care: one-vs-rest accuracy counts true negatives,
#  so a class the model almost never predicts still scores high. Running reaches
#  0.994 accuracy on an F1 of 0.106 - the model gets it right by rarely guessing
#  Running at all. F1 and balanced accuracy are the honest columns here.
# ============================================================================
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory

S1, S2, S3 = "#2a78d6", "#eb6834", "#1baf7a"      # categorical slots 1-3
SURFACE, INK, INK_2, INK_3 = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8983"
GRID = "#e6e5e1"

SERIES = [("F1 score", f1, S1), ("Accuracy (one-vs-rest)", ovr_acc, S2),
          ("Balanced accuracy (one-vs-rest)", ovr_bal, S3)]
CLASS_SHORT = ["Lying\ndown", "Sitting", "Walking", "Running",
               "Bicycling", "Standing\nin place", "Standing &\nmoving"]

mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "figure.dpi": 110})

fig, ax = plt.subplots(figsize=(11.6, 6.4))
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)
fig.subplots_adjust(top=0.775, left=0.065, right=0.985, bottom=0.185)

x = np.arange(N_CLASSES)
w = 0.265
for n, (label, vals, colour) in enumerate(SERIES):
    ax.bar(x + (n - 1) * w, vals, width=w * 0.93, color=colour,
           label=label, zorder=3, linewidth=0)

# direct-label the headline series only
for xi, v in zip(x - w, f1):
    ax.text(xi, v + 0.022, f"{v:.2f}", ha="center", va="bottom",
            fontsize=9, color=INK_2, zorder=4)

ax.axhline(POOLED["macro_f1"], color=S1, linewidth=1.2, linestyle=(0, (5, 4)),
           alpha=.55, zorder=2)
ax.set_xlim(-0.58, N_CLASSES - 0.16)          # right margin for the reference label
ax.text(N_CLASSES - 0.50, POOLED["macro_f1"], f"macro-F1\n{POOLED['macro_f1']:.3f}",
        fontsize=8.6, color=S1, ha="left", va="center", linespacing=1.4)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_SHORT, fontsize=9.8, color=INK_2, linespacing=1.45)
# support counts on one fixed baseline, so they align regardless of label wrapping
_bt = blended_transform_factory(ax.transData, ax.transAxes)
for xi, c in zip(x, support):
    ax.text(xi, -0.135, f"{c:,}", transform=_bt, ha="center", va="top",
            fontsize=8.8, color=INK_3)
ax.set_ylim(0, 1.06)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_yticklabels([f"{v:.1f}" for v in np.arange(0, 1.01, 0.2)],
                   fontsize=9, color=INK_3)
ax.tick_params(axis="both", length=0, pad=7)
ax.set_axisbelow(True)
ax.yaxis.grid(True, color=GRID, linewidth=1, zorder=0)
ax.xaxis.grid(False)
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xlabel("Activity class  ·  test segments", fontsize=10.5,
              color=INK_2, labelpad=30)

leg = ax.legend(loc="lower left", bbox_to_anchor=(0, 1.015), ncol=3,
                frameon=False, fontsize=10, handlelength=1.1,
                handleheight=1.1, borderpad=0, columnspacing=2.2)
for t in leg.get_texts():
    t.set_color(INK_2)

fig.text(0.065, 0.965, "Random Forest — per-class performance",
         fontsize=16, color=INK, fontweight="bold", va="top")
fig.text(0.065, 0.918,
         "Pooled over 5 folds. One-vs-rest accuracy counts true negatives, so a class "
         "the model rarely predicts still scores high —\nRunning reaches 0.99 accuracy "
         "on an F1 of 0.11. F1 and balanced accuracy are the columns that carry meaning.",
         fontsize=9.6, color=INK_3, va="top", linespacing=1.5)

out = os.path.join(FIG_DIR, "per_class_metrics.png")
fig.savefig(out, dpi=200, facecolor=SURFACE)
print(f"saved {out}  ({os.path.getsize(out)/1000:.0f} KB)")
plt.close(fig)